In [9]:
import pandas as pd
from pathlib import Path
import sys
sys.path.append("../src")
from preprocessing import build_preprocessor

MAIN_DF_PATH = Path("../data/processed/main_df.csv")
main_df = pd.read_csv(MAIN_DF_PATH)

In [10]:
# look at the df
main_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,total_order_price,total_freight,num_items,total_payment_value,max_payment_installment,payment_type
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,29.99,8.72,1.0,38.71,1.0,Voucher
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,118.70,22.76,1.0,141.46,1.0,Boleto
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58,159.90,19.22,1.0,179.12,3.0,Credit Card
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,2017-12-05 19:21:58,45.00,27.20,1.0,72.20,1.0,Credit Card
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,NaN,NaN,2018-02-17 00:00:00,2018-02-18 13:02:51,19.90,8.72,1.0,28.62,1.0,Credit Card


In [11]:
# move target to end
col_to_move = main_df.pop("review_score")
main_df.insert(len(main_df.columns), "review_score", col_to_move)

In [12]:
# drop date columns
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date", "review_creation_date",
             "review_answer_timestamp"]
main_df = main_df.drop(columns=date_cols)

In [13]:
# drop na values from target
main_df = main_df.dropna(subset=["review_score"])

In [14]:
# split into training and test sets
from sklearn.model_selection import train_test_split

X = main_df.drop("review_score", axis=1)
y = main_df.review_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# select numerical and categorical features
num_features = X.select_dtypes(include="number").columns
cat_features = ["order_status", "customer_city", "customer_state", "payment_type"]

In [16]:
# create pipelines
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

preprocessor = build_preprocessor(num_features, cat_features)

pipelines = {
    "Multiple Linear Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("multiple linear regressor", LinearRegression())
    ]),
    "Random Forest Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("random forest regressor", RandomForestRegressor(n_estimators=100, random_state=42))
    ]),

    "SVR": Pipeline([
        ("preprocessor", preprocessor),
        ("support vector regressor", SVR(kernel="rbf"))
    ])
}


In [17]:
# train models and calculate metrics
import joblib
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

results = {}

for model_name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results[model_name] = {
        "r² score": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred)
    }
    
    filename = model_name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(pipeline, f"../outputs/models/{filename}")

In [18]:
# save metrics
model_metrics_df = pd.DataFrame(results).T.round(3)
model_metrics_df.to_csv("../data/processed/model_metrics.csv")